In [2]:
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [3]:
processedData = pd.read_csv('../data/processed/train_clientes_features_eda.csv')
print("Dataset preparado para modelado.")
print(processedData.head())

Dataset preparado para modelado.
   ID_CORRELATIVO  CODMES  FLG_BANCARIZADO  RANG_INGRESO  FLAG_LIMA_PROVINCIA  \
0           35653  201208                1             6                    1   
1           66575  201208                1             3                    2   
2           56800  201208                1             1                    2   
3            8410  201208                1             4                    2   
4            6853  201208                1             0                    1   

   EDAD  ANTIGUEDAD  ATTRITION  RANG_SDO_PASIVO_MENOS0  SDO_ACTIVO_MENOS0  \
0  25.0         6.0          0                      14                  0   
1  27.0         0.0          0                       1                  0   
2  34.0         4.0          0                       7                  0   
3  63.0         5.0          0                       8                  0   
4  25.0         0.0          0                       1                  0   

   ...  FLG_SDO_O

In [4]:
df = processedData.copy()

In [5]:
if 'ATTRITION' not in df.columns:
    import numpy as np
    df['ATTRITION'] = np.random.randint(0,2,len(df))

In [6]:
# Variables para modelar
X = df.drop(columns=['ID_CORRELATIVO', 'CODMES', 'ATTRITION'])
y = df['ATTRITION']

In [7]:
# Encoding categóricas
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].fillna('Desconocido'))

In [8]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

In [9]:
# Convierte TODAS las columnas enteras a float64 (por seguridad)
for col in X_train.columns:
    if pd.api.types.is_integer_dtype(X_train[col]):
        X_train[col] = X_train[col].astype('float64')
        X_test[col] = X_test[col].astype('float64')

In [10]:
# --- MLflow experiment ---
import mlflow

mlflow.set_tracking_uri("arn:aws:sagemaker:us-west-1:880138931512:mlflow-tracking-server/mlflow-app") 
#mlflow.set_tracking_uri("sagemaker") 
#mlflow.set_tracking_uri("file:///home/sagemaker-user/mlruns")
mlflow.set_experiment("UTEC_Bank_Attrition")

with mlflow.start_run():
    # Parámetros
    n_estimators = 100
    max_depth = 5

    # Modelo
    clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    clf.fit(X_train, y_train)

    # Predicciones
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    # Métricas
    acc = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    # Log params y métricas
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("roc_auc", roc_auc)

    # Crear input_example y signature
    input_example = X_test.iloc[:1].copy() 
    signature = infer_signature(X_test, y_proba)

    # Log modelo con input_example y signature
    mlflow.sklearn.log_model(
        sk_model=clf,
        artifact_path="random_forest_model",
        input_example=input_example,
        signature=signature
    )

    print(f"Accuracy: {acc:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")

UnsupportedModelRegistryStoreURIException:  Model registry functionality is unavailable; got unsupported URI 'arn:aws:sagemaker:us-west-1:880138931512:mlflow-tracking-server/mlflow-app' for model registry data storage. Supported URI schemes are: ['', 'file', 'databricks', 'databricks-uc', 'uc', 'http', 'https', 'postgresql', 'mysql', 'sqlite', 'mssql']. See https://www.mlflow.org/docs/latest/tracking.html#storage for how to run an MLflow server against one of the supported backend storage locations.

In [11]:
import mlflow

print("Current tracking URI:", mlflow.get_tracking_uri())

Current tracking URI: arn:aws:sagemaker:us-west-1:880138931512:mlflow-tracking-server/mlflow-app
